In [1]:
%pip install numpy pandas seaborn torch torchvision scikit-learn matplotlib graphviz

Note: you may need to restart the kernel to use updated packages.


In [21]:
from sklearn.datasets import load_iris
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# K-Means is unsupervised, so we won't use train_test_split here
iris = load_iris()
X, y = iris.data, iris.target


# Show classes
print("Features:", iris.feature_names)
print("Classes:", iris.target_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2).fit_transform(X_scaled)

X_scaled = pca

Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: ['setosa' 'versicolor' 'virginica']


In [39]:
from sklearn.cluster import AgglomerativeClustering

# linkage: {‘ward’, ‘complete’, ‘average’, ‘single’}, default=’ward’

agglo = AgglomerativeClustering(
    metric="euclidean", linkage="ward", n_clusters=3
)

agglo.fit(X_scaled)

labels = agglo.labels_
n_clusters = agglo.n_clusters_

In [40]:
from sklearn.metrics import accuracy_score
from scipy.stats import mode
import numpy as np


def cluster_acc(y_true, y_pred):
    labels = np.zeros_like(y_pred)
    for i in np.unique(y_pred):
        mask = y_pred == i
        labels[mask] = mode(y_true[mask])[0]
    return accuracy_score(y_true, labels)


acc = cluster_acc(y, labels)
print("Cluster accuracy:", acc)


Cluster accuracy: 0.8066666666666666


# Scratch

In [ ]:
import numpy as np


class AgglomerativeClusteringScratch:
    def __init__(self, n_clusters=None, linkage="average"):
        # Use euclidean matric for computing pairwise distance matrix
        self.n_clusters = n_clusters

        if linkage == "complete":
            self.linkage_distance_func = self.max_linkage_distance
        elif linkage == "single":
            self.linkage_distance_func = self.min_linkage_distance
        elif linkage == "average":
            self.linkage_distance_func = self.avg_linkage_distance
        elif linkage == "ward":
            self.linkage_distance_func = self.ward_method_distance

    def avg_linkage_distance(self, cluster_A, cluster_B):
        # Compute average linkage distance between two clusters
        distance = 0
        for i in range(cluster_A.shape[0]):
            distance += np.linalg.norm(cluster_B - cluster_A[i, :], axis=1).sum()
        distance /= cluster_A.shape[0] * cluster_B.shape[0]
        return distance

    def max_linkage_distance(self, cluster_A, cluster_B):
        # Compute maximum linkage distance between two clusters
        distance = 0
        for i in range(cluster_A.shape[0]):
            distance = np.append(
                np.linalg.norm(cluster_B - cluster_A[i, :], axis=1), distance
            ).max()
        return distance

    def min_linkage_distance(self, cluster_A, cluster_B):
        # Compute minimum linkage distance between two clusters
        distance = np.inf
        for i in range(cluster_A.shape[0]):
            distance = np.append(
                np.linalg.norm(cluster_B - cluster_A[i, :], axis=1), distance
            ).min()
        return distance

    def ward_method_distance(self, cluster_A, cluster_B):
        # Compute the Ward linkage distance between two clusters
        n_A = cluster_A.shape[0]
        n_B = cluster_B.shape[0]

        centroid_A = np.mean(cluster_A, axis=0)
        centroid_B = np.mean(cluster_B, axis=0)

        # Distance is proportional to the squared Euclidean distance between centroids
        # scaled by size of the clusters
        diff = centroid_A - centroid_B
        distance = (n_A * n_B) / (n_A + n_B) * np.dot(diff, diff)

        return distance

    def pairwise_distance(self, data, n_samples):
        # Compute the pairwise distance matrix in euclidean matric
        distance_mat = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(i + 1, n_samples):
                distance = np.linalg.norm(data[i] - data[j])
                distance_mat[i, j] = distance
                distance_mat[j, i] = distance
        return distance_mat

    def update(self, data, distance_mat, labels):
        # "Find closest clusters, merge clusters, delete cluster, update distance"
        idx_upper = np.triu_indices(
            distance_mat.shape[0], k=1
        )  # Index of upper part of distance matrix (skip diagonal)
        min_value = np.min(distance_mat[idx_upper])  # Value of idx_upper
        row, col = np.argwhere(distance_mat == min_value)[
            0
        ]  # Index of min_value (same as d_kl)

        # Update label
        labels[labels == col] = row
        labels[labels > col] -= 1

        # Deleted the row and column 'col'
        distance_mat = np.delete(distance_mat, col, 0)
        distance_mat = np.delete(distance_mat, col, 1)

        # Update distance matrix
        for i in range(len(distance_mat)):
            distance_mat[row, i] = self.linkage_distance_func(
                data[labels == row], data[labels == i]
            )
            distance_mat[i, row] = distance_mat[row, i]
        return distance_mat, labels

    def fit_predict(self, X):
        self.data = X
        self.n_samples = self.data.shape[0]
        self.initial_distance = self.pairwise_distance(self.data, self.n_samples)
        self.labels = np.arange(self.n_samples)
        self.distance_matrix = self.initial_distance.copy()
        while len(np.unique(self.labels)) > self.n_clusters:
            # Fill in the diagonal as infinity to determine that the distance is the same position.
            np.fill_diagonal(self.distance_matrix, np.inf)
            self.distance_matrix, self.labels = self.update(
                self.data, self.distance_matrix, self.labels
            )

        return self.labels

In [42]:
from sklearn.metrics import accuracy_score

ACS_avg = AgglomerativeClusteringScratch(n_clusters=3, linkage='ward')
y_pred = ACS_avg.fit_predict(X_scaled)

acc =accuracy_score(y, y_pred)
print("Cluster accuracy:", acc)

Cluster accuracy: 0.7666666666666667
